In [1]:
from src.clients.gcp import get_bucket
from src.clients.snowflake import get_snowflake_connection
from src.config.settings import *

bucket = get_bucket(GCP_BUCKET)
print(bucket)

conn = get_snowflake_connection()
print("Snowflake connected")


/opt/homebrew/Caskroom/miniconda/base/envs/p_fintech/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/opt/homebrew/Caskroom/miniconda/base/envs/p_fintech/lib/python3.10/site-packages/snowflake/connector/vendored/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0.post1)/charset_normalizer (None) doesn't match a supported version!
  warnings.warn(


<Bucket: fintech-fraud-data-sandrade_portfolio>
Snowflake connected


In [2]:
key_path = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")
key_path

'/Users/seleneandrade/Documents/portfolio/paysim-fraud-analytics-platform/secrets/fraud-loader-key.json'

In [2]:
from src.clients.spark_builder import get_spark

In [3]:
spark = get_spark()


26/04/22 12:56:00 WARN Utils: Your hostname, MacBook-Pro-de-Selene.local resolves to a loopback address: 127.0.0.1; using 192.168.1.146 instead (on interface en0)
26/04/22 12:56:00 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/opt/homebrew/Caskroom/miniconda/base/envs/p_fintech/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/seleneandrade/.ivy2/cache
The jars for the packages stored in: /Users/seleneandrade/.ivy2/jars
net.snowflake#spark-snowflake_2.12 added as a dependency
net.snowflake#snowflake-jdbc added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d0d088d6-bd32-45c2-96b2-8d0eaa768064;1.0
	confs: [default]
	found net.snowflake#spark-snowflake_2.12;3.1.1 in central
	found net.snowflake#snowflake-jdbc;3.19.0 in central
	found org.apache.parquet#parquet-avro;1.13.1 in central
	found org.apache.parquet#parquet-column;1.13.1 in central
	found org.apache.parquet#parquet-common;1.13.1 in central
	found org.apache.parquet#parquet-format-structures;1.13.1 in central
	found org.slf4j#slf4j-api;1.7.22 in central
	found org.apache.parquet#parquet-encoding;1.13.1 in central
	found org.apache.yetus#audience-annotations;0.13.0 in central
	found org.apache.parquet#parquet-hadoop;1.13.1 in central
	found org.xerial.snappy#snappy-java;1.1.8.3 in centr

In [12]:
from pyspark.sql.functions import current_date, current_timezone
from src.clients.snowflake import get_snowflake_spark_options
class GCPDataReader:
    def __init__(self, spark, bucket, gcp_table_name, snowflake_table_name, format="csv", snowflake_schema=RAW_SNOWFLAKE_SCHEMA):
        self.bucket = bucket
        self.spark = spark
        self.gcp_table_name = gcp_table_name
        self.snowflake_table_name = snowflake_table_name
        self.error_path = f"gs://{self.bucket.name}/landing/error/{self.gcp_table_name}/"
        self.format = format
        self.snowflake_schema = snowflake_schema


    def read_data_from_gcp(self):
        blobs = self.bucket.list_blobs(prefix=f'landing/incoming/{self.gcp_table_name}')
        file_paths = list(set(f"gs://{self.bucket.name}/{blob.name}"   for blob in blobs if not blob.name.endswith(f"/")))
        return file_paths

    def get_dataframe_from_inbound(self, inbound_files, landing_schema=None):

        if not inbound_files:
            return None
        else:
            if landing_schema:
                df = (
                    self.spark.read.format(self.format)
                    .schema(landing_schema)
                    .options(header="true")
                    .option("badRecordsPath", self.error_path)
                    .load(inbound_files)
                )
            else:
                df = (
                    self.spark.read.format(self.format)
                    .options(header="true", inferSchema="true")
                    .option("badRecordsPath", self.error_path)
                    .load(inbound_files)
                )
            return df

    def move_inbound_files_to_processed(self, inbound_files):
        for file_path in inbound_files:

            blob_name = file_path.replace(f"gs://{self.bucket.name}/", "")
            source_blob = self.bucket.blob(blob_name)

            target_blob_name = blob_name.replace(
                "landing/incoming/",
                "landing/processed/",
                1
            )

            self.bucket.copy_blob(
                source_blob,
                self.bucket,
                target_blob_name
            )

            source_blob.delete()

    # def move_inbound_files_to_processed(self, df):
    #     processed_path = f"gs://{self.bucket.name}/landing/processed/{self.table_name}/"
    #     df.write.format("parquet").mode("append").option("header", "true").save(processed_path)

    def append_data_to_snowflake(self, df):
        if df is not None:
            sf_options = get_snowflake_spark_options()
            sf_options.update({"sfSchema": self.snowflake_schema})
            df.write \
                .format("snowflake") \
                .options(**sf_options) \
                .option("dbtable", self.snowflake_table_name) \
                .mode("append") \
                .save()

    def read_data_from_landing(self, landing_schema=None):
        inbound_files = self.read_data_from_gcp()
        df = self.get_dataframe_from_inbound(inbound_files, landing_schema)
        df = df.withColumn("_ingestion_date", current_date()).withColumn("_raw_ingestion_timestamp", current_timezone())

        self.append_data_to_snowflake(df)
        self.move_inbound_files_to_processed(inbound_files)
        return df



In [13]:
table_name = "paysim"

data_reader = GCPDataReader(spark, bucket, table_name)
df = data_reader.read_data_from_landing()

In [14]:
df.count()

321384

In [15]:
with open(SNOWFLAKE_PRIVATE_KEY_PATH, "r") as f:
    private_key = f.read()

sfOptions = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER,
    "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE,
    "sfSchema": RAW_SNOWFLAKE_SCHEMA,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE
}

In [19]:
sfOptions.update({"sfSchema": "test"})

In [20]:
print(sfOptions)

{'sfURL': 'ey04584.us-east-2.aws.snowflakecomputing.com', 'sfUser': 'SELENEANDRADE0922', 'sfPassword': 'Snowflake2026*', 'sfDatabase': 'FRAUD_DB', 'sfSchema': 'test', 'sfWarehouse': 'FRAUD_WH'}


In [17]:
df.write \
    .format("snowflake") \
    .options(**sfOptions) \
    .option("dbtable", "TRANSACTIONS_RAW") \
    .mode("append") \
    .save()

In [10]:
import pandas as pd

query = """
select *
from TRANSACTIONS_RAW
limit 100
"""

df = pd.read_sql(query, conn)
df.head()

/var/folders/00/6v4ypdjx7mlb7kcwyj2mg58m0000gn/T/ipykernel_38109/2861366898.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,STEP,TYPE,AMOUNT,NAMEORIG,OLDBALANCEORG,NEWBALANCEORIG,NAMEDEST,OLDBALANCEDEST,NEWBALANCEDEST,ISFRAUD,ISFLAGGEDFRAUD,_INGESTION_DATE,_RAW_INGESTION_TIMESTAMP
0,11,CASH_IN,379680.19,C78296391,213493.00,593173.19,C2009999799,23365.00,0.00,0,0,2026-04-22,Europe/Madrid
1,11,CASH_OUT,359735.55,C1629496249,593173.19,233437.65,C1373074936,402724.69,762460.23,0,0,2026-04-22,Europe/Madrid
2,11,CASH_OUT,385173.18,C546807366,165428.00,0.00,C1806661928,313593.80,646241.96,0,0,2026-04-22,Europe/Madrid
3,11,CASH_OUT,301016.68,C149429480,6956.00,0.00,C936564798,0.00,730018.39,0,0,2026-04-22,Europe/Madrid
4,11,CASH_OUT,161215.08,C1505329215,174.00,0.00,C2135669532,534.11,232732.11,0,0,2026-04-22,Europe/Madrid


In [7]:
blobs = bucket.list_blobs(prefix=f'landing/incoming/{table_name}')
files = [f"gs://{bucket.name}/{blob.name}"   for blob in blobs if not blob.name.endswith(f"/")]
files

['gs://fintech-fraud-data-sandrade_portfolio/landing/incoming/paysim/batch_0001_161501rows.csv',
 'gs://fintech-fraud-data-sandrade_portfolio/landing/incoming/paysim/batch_0002_159883rows.csv']

In [12]:
df = spark.read.csv(
    files,
    header=True
)
df.count() #161501 159883

321384

In [13]:
161501+159883

321384

In [10]:
df = spark.read.csv(
    "gs://fintech-fraud-data-sandrade_portfolio/landing/incoming/",
    header=True
)

df.show(5)

AnalysisException: [UNABLE_TO_INFER_SCHEMA] Unable to infer schema for CSV. It must be specified manually.

In [ ]:
table_name = "paysim"
# blobs = bucket.list_blobs(prefix=f'landing/incoming/{table_name}')
# files = [blob for blob in blobs if blob.name.split("/")[-1] != "/"]

for data in bucket.list_blobs(prefix=f'landing/incoming/{table_name}'):
    #print(help(data))
    print(data.name.split("/")[-1])

In [ ]:
df = spark.read.csv(
    "gs://fintech-fraud-data-sandrade_portfolio/landing/incoming/",
    header=True
)

df.show(5)

In [ ]:
def get_incoming_files(bucket):
    blobs = bucket.list_blobs(prefix=INCOMING_PREFIX)
    return [blob for blob in blobs if blob.name.endswith(".csv")]

file_path = get_incoming_files(bucket)

In [ ]:
file_path

In [ ]:
df = spark.read.option("header", True).csv(file_path)

In [21]:
321384 - 319384

2000